# 01 | Geração de dados

**Pergunta de negócio:** temos uma base representativa o suficiente para investigar desempenho de vendas, produtos, regiões e relacionamento com clientes?

Este notebook cria um cenário sintético, reproduzível e sem dados pessoais reais. A semente fixa permite repetir análises e comparar resultados.

In [ ]:
from pathlib import Path; import pandas as pd; import sys
root = Path.cwd().resolve(); assert (root / 'src').is_dir(), 'Abra o notebook a partir da raiz do projeto'; sys.path.insert(0, str(root))

In [ ]:
from src.generate_data import generate_dataset
tables = generate_dataset(seed=42); raw_dir = root / 'data' / 'raw'; raw_dir.mkdir(parents=True, exist_ok=True)
[frame.to_csv(raw_dir / f'{name}.csv', index=False, encoding='utf-8') for name, frame in tables.items()]
{name: f'{len(frame):,} linhas x {len(frame.columns)} colunas' for name, frame in tables.items()}

## Escopo do dataset

O conjunto cobre 2023 a 2025, com dimensões de clientes, produtos, vendedores e regiões, além de uma tabela de itens de venda. Cada pedido pode conter mais de um item, então ticket médio deve ser calculado por `order_id`, não pela quantidade de linhas.

In [ ]:
pd.concat([frame.head(3).assign(tabela=name) for name, frame in tables.items()], ignore_index=True, sort=False)

In [ ]:
sales = tables['sales']; checks = {'cliente válido': sales.customer_id.isin(tables['customers'].customer_id).all(), 'produto válido': sales.product_id.isin(tables['products'].product_id).all(), 'vendedor válido': sales.seller_id.isin(tables['sellers'].seller_id).all(), 'região válida': sales.region_id.isin(tables['regions'].region_id).all()}; print(checks); print(f"Base gerada com {len(sales):,} itens em {sales.order_id.nunique():,} pedidos.")

## Resposta

A base combina **14 mil+ itens**, **7 mil pedidos**, recorrência de clientes e dimensões suficientes para comparar mix, regiões e vendedores. A cobertura das chaves é conferida acima; as medidas financeiras ainda serão derivadas na etapa de limpeza.